# Using biomed.sm.mv-te-84m Models Via SmallMoleculeMultiViewModel API

In [1]:
# Necessary Imports
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel, PredictionIterator
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.api.dataset_registry import DatasetRegistry

#imports for new BiomedMultiViewEncoder class
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline

from dataclasses import asdict
from itertools import islice
import pandas as pd
import os
import torch
import torch.nn as nn

#imports for visualization
import numpy as np
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
import seaborn as sns

/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torch_geometric/typing.py:18: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: dlopen(/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/libpyg.so, 0x0006): tried: '/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/libpyg.so' (mach-o file, but is an incompatible architecture (have 'x86_64', need 'arm64')), '/System/Volumes/Preboot/Cryptexes/OS/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/libpyg.so' (no such file), '/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/libpyg.so' (mach-o file, but is an incompatible architecture (have 'x86_64', need 'arm64'))
  warnings.warn(f"An issue occurred while importing 'pyg-li

In [2]:
from torch.nn.functional import pad
from torch.nn.utils.rnn import pad_sequence

class BiomedMultiViewEncoder(nn.Module):
    def __init__(
        self
    ):
        super(BiomedMultiViewEncoder, self).__init__()
        # Initialize the pretrained model
        biomed_smmv_pretrained = SmallMoleculeMultiViewModel.from_pretrained(
            LateFusionStrategy.ATTENTIONAL,
            #model_path='../data_root/bmfm_model_dir/pretrained/MULTIVIEW_MODEL/biomed-smmv-with-coeff-agg.pth',
            inference_mode=False,
        )
        # Initialize the model subcomponents
        self.model_graph = biomed_smmv_pretrained.model_graph # output dim: 512
        self.model_image = biomed_smmv_pretrained.model_image # output dim: 512
        self.model_text = biomed_smmv_pretrained.model_text   # output dim: 768

    # Helper function for collating the individual processed graph samples:
    @staticmethod
    def collate_graph_data(graph_data_list):
        collated = {}
        collated["node_num"] = torch.cat([sample['node_num'] for sample in graph_data_list])
        collated["node_data"] = torch.cat([sample['node_data'] for sample in graph_data_list])
        collated["edge_num"] = torch.cat([sample['edge_num'] for sample in graph_data_list])
        collated["edge_data"] = torch.cat([sample['edge_data'] for sample in graph_data_list])
        collated["edge_index"] = torch.cat([sample['edge_index'] for sample in graph_data_list], dim=1)

        max_node_num = max(collated["node_num"])
        collated["lap_eigvec"] = torch.cat(
                [
                    pad(i, (0, max_node_num - i.size(1)), value=float("0"))
                    for i in [sample["lap_eigvec"] for sample in graph_data_list]
                ]
            )
        return collated

    def forward(self, smiles: list):
        tokenized_smiles_list = []
        attention_mask_list = []
        graph_data_list = []
        
        image_tensors = []
        graph_emb = []

        for sm in smiles:
            # Prepare image and text data in batch format
            img_data = ImageFinetuneDataPipeline.smiles_to_image_format(sm)
            image_tensors.append(img_data['img'].squeeze(0)) # Remove extra batch dimension if present

            txt_data = TextFinetuneDataPipeline.smiles_to_text_format(sm)
            tokenized_smiles_list.append(txt_data['smiles.tokenized'].squeeze(0))
            attention_mask_list.append(txt_data['attention_mask'].squeeze(0))

            # Run the graph model on individual smiles
            graph_data = Graph2dFinetuneDataPipeline.smiles_to_graph_format(sm)
            graph_data_list.append(graph_data)
            #graph_emb.append(self.model_graph(graph_data).squeeze(0))

        # Run the image and text models on the batched data
        image_batch = torch.stack(image_tensors, dim=0) #.to(device)
        tokenized_smiles_batch = pad_sequence(tokenized_smiles_list, batch_first=True) #.to(device)
        attention_mask_batch = pad_sequence(attention_mask_list, batch_first=True) #.to(device)
        graph_batch = BiomedMultiViewEncoder.collate_graph_data(graph_data_list)
        
        image_emb = self.model_image(image_batch)
        text_emb = self.model_text(tokenized_smiles_batch, attention_mask_batch)
        graph_emb = self.model_graph(graph_batch)

        return [graph_emb, image_emb, text_emb]

## Explore the data

Following are the datasets available for evaluation, finetuning and inference.

In [3]:
pd.DataFrame(DatasetRegistry.get_instance().get_collection('MoleculeNet'))

,dataset_name,num_tasks,task_type,description,preferred_metric,path,example,collection,num_classes
0,BACE,1,TaskType.CLASSIFICATION,MoleculeNet: Inhibition of human beta secretase 1,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/bace.csv,"CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,0",DatasetCollection.MOLECULENET,2.0
1,BBBP,1,TaskType.CLASSIFICATION,MoleculeNet: Blood brain barrier penetration,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/bbbp.csv,"CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,0",DatasetCollection.MOLECULENET,2.0
2,CLINTOX,2,TaskType.CLASSIFICATION,MoleculeNet: Toxicity data of FDA-approved dru...,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/clintox.csv,"[N+](=O)([O-])[O-],1 0",DatasetCollection.MOLECULENET,2.0
3,ESOL,1,TaskType.REGRESSION,MoleculeNet: Water solubility data for organics,Metrics.RMSE,datasets/raw_data/MoleculeNet/esol.csv,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,DatasetCollection.MOLECULENET,NaN
4,FREESOLV,1,TaskType.REGRESSION,MoleculeNet: Hydration free energy,Metrics.RMSE,datasets/raw_data/MoleculeNet/freesolv.csv,"CN(C)C(=O)c1ccc(cc1)OC,-11.01",DatasetCollection.MOLECULENET,NaN
5,HIV,1,TaskType.CLASSIFICATION,MoleculeNet: Inhibition of HIV viral replication,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/hiv.csv,"CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,0",DatasetCollection.MOLECULENET,2.0
6,LIPOPHILICITY,1,TaskType.REGRESSION,MoleculeNet: Octonol/water distribution coefff...,Metrics.RMSE,datasets/raw_data/MoleculeNet/lipophilicity.csv,"Cn1c(CN2CCN(CC2)c3ccc(Cl)cc3)nc4ccccc14,3.54",DatasetCollection.MOLECULENET,NaN
7,MUV,17,TaskType.CLASSIFICATION,MoleculeNet: PubChem derived target-activity t...,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/muv.csv,Cc1cccc(N2CCN(C(=O)C34CC5CC(CC(C5)C3)C4)CC2)c1...,DatasetCollection.MOLECULENET,2.0
8,PCBA,128,TaskType.CLASSIFICATION,MoleculeNet: PubChem derived target-activity (...,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/pcba.csv,"CC(=O)N1CCC2(CC1)NC(=O)N(c1ccccc1)N2,0 0 -1 0 ...",DatasetCollection.MOLECULENET,2.0
9,QM7,1,TaskType.REGRESSION,MoleculeNet: Electronic properties derived fro...,Metrics.MAE,datasets/raw_data/MoleculeNet/qm7.csv,"C([H])([H])([H])[H],-417.96",DatasetCollection.MOLECULENET,NaN


Get more information about a particular dataset

In [4]:
dataset = DatasetRegistry.get_instance().get_dataset_info('TOX21')
pd.DataFrame([asdict(dataset)])

,dataset_name,num_tasks,task_type,description,preferred_metric,path,example,collection,num_classes
0,TOX21,12,TaskType.CLASSIFICATION,MoleculeNet: Toxicity against set of targets,Metrics.ROCAUC,datasets/raw_data/MoleculeNet/tox21.csv,"CCOc1ccc2nc(S(N)(=O)=O)sc2c1,0 0 1 -1 -1 0 0 1...",DatasetCollection.MOLECULENET,2


: 

In [ ]:
#New BiomedMultiViewEncoder class
os.environ['BMFM_HOME'] = '/Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny'

def get_all_moleculenet_smiles():
    dataset_registry = DatasetRegistry()
    all_smiles = set()  # Using set to avoid duplicates
    
    # Get all MoleculeNet datasets
    moleculenet_datasets = dataset_registry.get_collection('MoleculeNet')
    
    for dataset_info in moleculenet_datasets:
        dataset_name = dataset_info.dataset_name  # Access as attribute, not dict
        ds = dataset_registry.get_dataset_info(dataset_name)
        
        try:
            # Read the CSV file
            df = pd.read_csv('/Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/datasets/raw_data/MoleculeNet/esol.csv')
            
            # The first column is typically SMILES
            smiles_col = df.columns[1]
            all_smiles.update(df[smiles_col].tolist())
            
        except Exception as e:
            print(f"Error processing dataset {dataset_name}: {str(e)}")
            continue
    
    return list(all_smiles)

# Get all unique SMILES
all_smiles = get_all_moleculenet_smiles()
print(f"Total unique SMILES: {len(all_smiles)}")

# Optional: Look at a few examples
print("\nExample SMILES:")
for smile in list(all_smiles)[:5]:
    print(smile)

encoder = BiomedMultiViewEncoder()
smiles_list = list(all_smiles)
embeddings = encoder(smiles_list)
print(embeddings)

# Access individual embeddings
graph_embeddings = embeddings[0]  # Shape: [batch_size, 512]
image_embeddings = embeddings[1]  # Shape: [batch_size, 512]
text_embeddings = embeddings[2]   # Shape: [batch_size, 768]

print(graph_embeddings.shape)
print(image_embeddings.shape)
print(text_embeddings.shape)

Total unique SMILES: 1123

Example SMILES:
Clc1ccc(cc1)c2c(Cl)cccc2Cl
Nc1cccc(c1)N(=O)=O
COP(=S)(OC)Oc1cc(Cl)c(Br)cc1Cl
Cc1ccc(cc1)N(=O)=O
CCCCCCCCBr


/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2025-02-07 11:06:17,155 - root - INFO - MacBook-Pro-231.stdusr.yale.internal:7990824960:0:0 - Using coeff_mlp architecture for aggregator
2025-02-07 11:06:17,155 - root - INFO - MacBook-Pro-231.stdusr.yale.internal:7990824960:0:0 - dim_list [512, 512, 768] for aggregator
2025-02-07 11:06:17,163 - root - INFO - MacBook-Pro-231.stdusr.yale.internal:79908249

## Using Models from HuggingFace

We have made pretrained and finetuned models available in [HuggingFace](https://huggingface.co/ibm/biomed.sm.mv-te-84m)

### Loading Pretrained Model from HuggingFace

Load a pretrained model from Huggingface by setting `model_path` to a Huggingface repo, and setting the `huggingface` argument to `True`

In [6]:
model = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL,
                                                    model_path='ibm/biomed.sm.mv-te-84m',
                                                    huggingface=True)

2025-01-30 14:01:28,245 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m
/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2025-01-30 14:01:29,290 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 14:01:29,290 - root - INFO - MacBook-Pro-231.local:7932416000:

### Get Embeddings from a Pretrained Model

In [7]:
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(
    smiles=example_smiles,
    model_path="ibm/biomed.sm.mv-te-84m",
    huggingface=True,
)
print(example_emb.shape)

2025-01-30 14:01:31,568 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m
2025-01-30 14:01:32,582 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 14:01:32,582 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 14:01:32,803 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


torch.Size([512])


### Load Finetuned Model from HuggingFace

In [8]:
dataset_registry = DatasetRegistry()
ds = dataset_registry.get_dataset_info('BACE')

In [9]:
finetuned_model_ds = SmallMoleculeMultiViewModel.from_finetuned(
    ds,
    model_path="ibm/biomed.sm.mv-te-84m-MoleculeNet-ligand_scaffold-BACE-101",
    inference_mode=True,
    huggingface=True
)

2025-01-30 14:01:37,424 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m-MoleculeNet-ligand_scaffold-BACE-101
2025-01-30 14:01:38,441 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 14:01:38,441 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 14:01:38,958 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


### Get Predictions from Finetuned Model

In [10]:
# Get predictions
prediction = SmallMoleculeMultiViewModel.get_predictions(
    example_smiles, ds, finetuned_model=finetuned_model_ds
)
print(prediction)

tensor(0, dtype=torch.int32)


## Using Local Data

You can also download the checkpoints and data to your system and use the API. Follow procedure in the README for setting up the data. Make sure that `BMFM_HOME` environment variable is pointing to the data_root.

In [32]:
os.environ['BMFM_HOME'] = "/Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny" 
print(os.environ['BMFM_HOME'])

/Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny


### Loading Pretrained Model (Local)

Load our provided pretrained checkpoint, which uses a Attentional Late Fusion strategy (default). It loads the model in inference mode.

In [33]:
model_local = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL, inference_mode = True)

2025-01-30 15:37:21,274 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 15:37:21,275 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:21,282 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint from default path /Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny/bmfm_model_dir/pretrained/MULTIVIEW_MODEL/biomed-smmv-with-coeff-agg.pth
2025-01-30 15:37:21,385 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading pretrain checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2025-01-30 15:37:21,391 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


We can also load with a different fusion strategy - Note, aggregator will not be pretrained but BaseModel will be.

In [34]:
model_local = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.MOE_WEIGHTED_CONCAT_PROJECTED, inference_mode = True)

2025-01-30 15:37:25,362 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using moe_weighted_concat architecture for aggregator
2025-01-30 15:37:25,362 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:25,365 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint from default path /Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny/bmfm_model_dir/pretrained/MULTIVIEW_MODEL/biomed-smmv-with-coeff-agg.pth
2025-01-30 15:37:25,439 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading pretrain checkpoint for SmallMoleculeMultiView Model - _IncompatibleKeys(missing_keys=['aggregator.weighted_concat_network.gating_network.fc.weight', 'aggregator.weighted_concat_network.gating_network.fc.bias'], unexpected_keys=['aggregator.w_before_mean.0.weight', 'aggregator.w_before_mean.0.bias', 'aggregator.w_before_mean.2.weight', 'aggregator.down_project.weight'

We can also provide a `custom_model` path to load from a specific checkpoint

In [35]:
ckpt_path = f"{os.environ['BMFM_HOME']}/bmfm_model_dir/pretrained/MULTIVIEW_MODEL/biomed-smmv-base.pth"
model_local = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.CONCAT, 
                                     model_path=ckpt_path,
                                     inference_mode = True)

2025-01-30 15:37:28,404 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using concat architecture for aggregator
2025-01-30 15:37:28,405 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:28,405 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint from provided path /Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny/bmfm_model_dir/pretrained/MULTIVIEW_MODEL/biomed-smmv-base.pth
2025-01-30 15:37:28,498 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading pretrain checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2025-01-30 15:37:28,503 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


We can set `model_path` to `False` if to load a non-pretrained model.

In [36]:
model_local = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.MOE_NOISED_WEIGHTED_CONCAT_BOTH_PROJECTED, 
                                     model_path=False,
                                     inference_mode = False)

2025-01-30 15:37:34,711 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using moe_noised_weighted_concat architecture for aggregator
2025-01-30 15:37:34,712 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:34,714 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Not using checkpoint for model initialization


### Get Embeddings from a Pretrained Model (Local)

In [37]:
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(example_smiles)
example_emb.shape

2025-01-30 15:37:37,363 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 15:37:37,364 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:37,371 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint from default path /Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny/bmfm_model_dir/pretrained/MULTIVIEW_MODEL/biomed-smmv-with-coeff-agg.pth
2025-01-30 15:37:37,437 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading pretrain checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2025-01-30 15:37:37,441 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


torch.Size([512])

### Load Finetuned Model (Local)

Choose a supported dataset for which finetuned checkpoint is available

In [38]:
#Example of a Classification Prediction
dataset_registry = DatasetRegistry()
bace_ds = dataset_registry.get_dataset_info('BACE')

In [39]:
finetuned_model_bace_local = SmallMoleculeMultiViewModel.from_finetuned(bace_ds, inference_mode = True)

2025-01-30 15:37:43,603 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 15:37:43,604 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:43,939 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2025-01-30 15:37:44,097 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading finetune checkpoint for Prediction Head: /Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny/bmfm_model_dir/finetuned/MULTIVIEW_MODEL/MoleculeNet/ligand_scaffold/BACE/best-101.ckpt
2025-01-30 15:37:44,114 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


In [40]:
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
bace_prediction = SmallMoleculeMultiViewModel.get_predictions(example_smiles, bace_ds, finetuned_model=finetuned_model_bace_local)
bace_prediction

tensor(0, dtype=torch.int32)

In [41]:
#Example of a Regression Prediction
esol_ds = dataset_registry.get_dataset_info('ESOL')
finetuned_model_esol_local = SmallMoleculeMultiViewModel.from_finetuned(esol_ds, inference_mode = True)
esol_prediction = SmallMoleculeMultiViewModel.get_predictions(example_smiles, esol_ds, finetuned_model=finetuned_model_esol_local)
esol_prediction

2025-01-30 15:37:50,211 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 15:37:50,211 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 15:37:50,378 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2025-01-30 15:37:50,513 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading finetune checkpoint for Prediction Head: /Users/ananya_krishna/Documents/GitHub/biomed-multiview/code/biomed-multi-view/data_root/package_tiny/bmfm_model_dir/finetuned/MULTIVIEW_MODEL/MoleculeNet/ligand_scaffold/ESOL/best-101.ckpt
2025-01-30 15:37:50,528 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - in train False setting deterministic_eval = True


tensor(-3.5531)

In [49]:
def get_embeddings_for_visualization(model, smiles_list, batch_size=32):
    graph_embs = []
    image_embs = []
    text_embs = []
    
    # Process in batches
    for i in range(0, len(smiles_list), batch_size):
        batch_smiles = smiles_list[i:i+batch_size]
        embeddings = model(batch_smiles)
        
        graph_embs.append(embeddings[0].detach().cpu().numpy())
        image_embs.append(embeddings[1].detach().cpu().numpy())
        text_embs.append(embeddings[2].detach().cpu().numpy())
    
    # Concatenate all batches
    graph_embs = np.concatenate(graph_embs, axis=0)
    image_embs = np.concatenate(image_embs, axis=0)
    text_embs = np.concatenate(text_embs, axis=0)
    
    return graph_embs, image_embs, text_embs

def visualize_embeddings(graph_embs, image_embs, text_embs, method='pca'):
    # Combine all embeddings
    all_embs = np.vstack([graph_embs, image_embs, text_embs])
    
    # Create labels
    labels = ['Graph'] * len(graph_embs) + ['Image'] * len(image_embs) + ['Text'] * len(text_embs)
    
    # Reduce dimensionality
    if method.lower() == 'pca':
        reducer = PCA(n_components=2)
    else:  # UMAP
        reducer = umap.UMAP(n_components=2, random_state=42)
    
    reduced_embs = reducer.fit_transform(all_embs)
    
    # Plot
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=reduced_embs[:, 0], y=reduced_embs[:, 1], 
                    hue=labels, alpha=0.6)
    plt.title(f'{method.upper()} visualization of embeddings')
    plt.show()

# Example usage:
# First initialize your model
encoder = BiomedMultiViewEncoder()

# Get a subset of SMILES for visualization (using first 1000 for example)
sample_smiles = smiles_list[:100]

# Get embeddings
graph_embs, image_embs, text_embs = get_embeddings_for_visualization(encoder, sample_smiles)

# Visualize using PCA
visualize_embeddings(graph_embs, image_embs, text_embs, method='pca')

# Visualize using UMAP
visualize_embeddings(graph_embs, image_embs, text_embs, method='umap')

/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/ananya_krishna/Documents/GitHub/biomed-multiview/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2025-01-30 16:07:11,354 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Using coeff_mlp architecture for aggregator
2025-01-30 16:07:11,355 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - dim_list [512, 512, 768] for aggregator
2025-01-30 16:07:11,363 - root - INFO - MacBook-Pro-231.local:7932416000:0:0 - Loading checkpoint from default path

NameError: name 'all_smiles' is not defined